# Feature engineering - early prototyping

Where the feature logic in `feature_engineering.py` started, before the
module existed. Works directly off `games_clean.csv` / `games_matched.csv`,
not the final `games_final.csv`.

In [1]:
import pandas as pd

df = pd.read_csv('../data/processed/games_clean.csv')

df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])

print(df.dtypes)

SEASON_ID                     int64
TEAM_ID                       int64
TEAM_ABBREVIATION               str
TEAM_NAME                       str
GAME_ID                       int64
GAME_DATE            datetime64[us]
MATCHUP                         str
WL                              str
MIN                         float64
PTS                         float64
FGM                         float64
FGA                         float64
FG_PCT                      float64
FG3M                        float64
FG3A                        float64
FG3_PCT                     float64
FTM                         float64
FTA                         float64
FT_PCT                      float64
OREB                        float64
DREB                        float64
REB                         float64
AST                         float64
STL                         float64
BLK                         float64
TOV                         float64
PF                          float64
PLUS_MINUS                  

## 1. Rest days and back-to-back

Days since a team's previous game in the season, and whether it's a
back-to-back (`rest_days == 1`).

In [2]:
df_sorted = df.sort_values(['TEAM_ABBREVIATION', 'GAME_DATE'])

df_sorted['rest_days'] = df_sorted.groupby(['TEAM_ABBREVIATION', 'SEASON_ID'])['GAME_DATE'].diff().dt.days

median_rest = df_sorted['rest_days'].median()
df_sorted['rest_days'] = df_sorted['rest_days'].fillna(median_rest)

df_sorted['is_back_to_back'] = (df_sorted['rest_days'] == 1).astype(int)

print(df_sorted[['TEAM_ABBREVIATION', 'SEASON_ID', 'GAME_DATE', 'rest_days', 'is_back_to_back']].head(15))

    TEAM_ABBREVIATION  SEASON_ID  GAME_DATE  rest_days  is_back_to_back
18                ATL      22020 2020-12-23        2.0                0
39                ATL      22020 2020-12-26        3.0                0
78                ATL      22020 2020-12-28        2.0                0
111               ATL      22020 2020-12-30        2.0                0
140               ATL      22020 2021-01-01        2.0                0
163               ATL      22020 2021-01-02        1.0                1
187               ATL      22020 2021-01-04        2.0                0
217               ATL      22020 2021-01-06        2.0                0
264               ATL      22020 2021-01-09        3.0                0
301               ATL      22020 2021-01-11        2.0                0
350               ATL      22020 2021-01-15        4.0                0
366               ATL      22020 2021-01-16        1.0                1
381               ATL      22020 2021-01-18        2.0          

In [3]:
print(df_sorted.value_counts('is_back_to_back'))

is_back_to_back
0    11952
1     2612
Name: count, dtype: int64


In [4]:
print(df_sorted[(df_sorted['TEAM_ABBREVIATION'] == 'ATL')].groupby('SEASON_ID').head(1)[
    ['SEASON_ID', 'GAME_DATE', 'rest_days']])

       SEASON_ID  GAME_DATE  rest_days
18         22020 2020-12-23        2.0
2187       22021 2021-10-21        2.0
4628       22022 2022-10-19        2.0
7085       22023 2023-10-25        2.0
9547       22024 2024-10-23        2.0
12013      22025 2025-10-22        2.0
14471      22026 2026-10-21        2.0


**Diverged from production.** Here, a missing `rest_days` (a team's first
game of the season) is filled with the league-wide median. The final
`add_rest_days` in `feature_engineering.py` leaves it as `NaN` instead - the
project settled on not imputing at all, letting XGBoost's native NaN
handling (and the separate `is_early_season` flag) do the work rather than
inventing a plausible-looking rest value for a game with no history behind
it.

## 2. Rolling averages

5-game rolling mean of box-score stats, per team per season, shifted by one
game so the current game's own result never leaks into its own feature.

In [5]:
stat_cols = ['PTS', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'FG_PCT', 'FG3_PCT']

window=5

for col in stat_cols:
    df_sorted[f'rolling_{col.lower()}_{window}'] = df_sorted.groupby(['TEAM_ABBREVIATION', 'SEASON_ID'])[col].shift(1).rolling(window=window, min_periods=1).mean()



In [6]:
print(df_sorted[['TEAM_ABBREVIATION', 'PTS', 'rolling_pts_5']].head(15))

    TEAM_ABBREVIATION    PTS  rolling_pts_5
18                ATL  124.0            NaN
39                ATL  122.0     124.000000
78                ATL  128.0     123.000000
111               ATL  141.0     124.666667
140               ATL  114.0     128.750000
163               ATL   91.0     125.800000
187               ATL  108.0     119.200000
217               ATL   94.0     116.400000
264               ATL  105.0     109.600000
301               ATL  112.0     102.400000
350               ATL   92.0     102.000000
366               ATL  106.0     102.200000
381               ATL  108.0     101.800000
406               ATL  123.0     104.600000
436               ATL  116.0     108.200000


**Made it into production unchanged.** `add_rolling_averages` in
`feature_engineering.py` is this exact `shift(1)` + `rolling(...).mean()`
pattern, just parameterised over both `window=5` and `window=10` and more
stat columns.

## 3. Elo

The core signal of the whole model — `elo_diff` and `elo_win_prob` end up
carrying most of it (see `notebooks/model_training.ipynb`). This is the
plain win/loss version, before any tuning.

In [7]:
def calculate_elo(matched_df, k=20, home_advantage=100, initial_elo=1500, season_regression=0.75):
    # matched_df: one row one match 
 
    ratings = {}  # dict: name of the team : current ranking
    home_elo_before = []  # rating before match
    away_elo_before = []
    current_season = None
 
    for idx, row in matched_df.iterrows():
        team_home = row['HOME_TEAM_ABBR']
        team_away = row['AWAY_TEAM_ABBR']
        season = row['SEASON_ID']
 
        # 1. season changed 
        if current_season is not None and season != current_season:
            for team in ratings:
                ratings[team] = season_regression * ratings[team] + (1 - season_regression) * initial_elo
        current_season = season
 
        # 2. load current rating (initial 1500)
        home_rating = ratings.get(team_home, initial_elo)
        away_rating = ratings.get(team_away, initial_elo)
 
        # 3. save rating
        home_elo_before.append(home_rating)
        away_elo_before.append(away_rating)
 
        # 4. expected score (with home bonus)
        expected_home = 1 / (1 + 10 ** ((away_rating - (home_rating + home_advantage)) / 400))
 
        # 5. actual score
        actual_home = row['home_win']
 
        # 6. updating ratings after match
        new_home_rating = home_rating + k * (actual_home - expected_home)
        new_away_rating = away_rating + k * ((1 - actual_home) - (1 - expected_home))
 
        ratings[team_home] = new_home_rating
        ratings[team_away] = new_away_rating
 
    matched_df['home_elo'] = home_elo_before
    matched_df['away_elo'] = away_elo_before
    return matched_df

In [8]:
matched = pd.read_csv("../data/processed/games_matched.csv")

matched = calculate_elo(matched)

print(matched[['GAME_DATE', 'HOME_TEAM_ABBR', 'AWAY_TEAM_ABBR', 'home_elo', 'away_elo', 'home_win']].head(30))

     GAME_DATE HOME_TEAM_ABBR AWAY_TEAM_ABBR    home_elo     away_elo  \
0   2020-12-22            BKN            GSW  1500.00000  1500.000000   
1   2020-12-22            LAL            LAC  1500.00000  1500.000000   
2   2020-12-23            POR            UTA  1500.00000  1500.000000   
3   2020-12-23            DEN            SAC  1500.00000  1500.000000   
4   2020-12-23            MEM            SAS  1500.00000  1500.000000   
5   2020-12-23            CHI            ATL  1500.00000  1500.000000   
6   2020-12-23            TOR            NOP  1500.00000  1500.000000   
7   2020-12-23            MIN            DET  1500.00000  1500.000000   
8   2020-12-23            ORL            MIA  1500.00000  1500.000000   
9   2020-12-23            IND            NYK  1500.00000  1500.000000   
10  2020-12-23            CLE            CHA  1500.00000  1500.000000   
11  2020-12-23            PHX            DAL  1500.00000  1500.000000   
12  2020-12-23            BOS            MIL  1500.

In [9]:
team = 'BOS'
team_games = matched[(matched['HOME_TEAM_ABBR'] == team) | (matched['AWAY_TEAM_ABBR'] == team)].copy()

team_games['team_elo'] = team_games.apply(
    lambda row: row['home_elo'] if row['HOME_TEAM_ABBR'] == team else row['away_elo'],
    axis=1
)

print(team_games.groupby('SEASON_ID').tail(3)[['GAME_DATE', 'SEASON_ID', 'team_elo']])
print("---")
print(team_games.groupby('SEASON_ID').head(3)[['GAME_DATE', 'SEASON_ID', 'team_elo']])

       GAME_DATE  SEASON_ID     team_elo
1036  2021-05-12      22020  1484.395477
1062  2021-05-15      22020  1472.752658
1071  2021-05-16      22020  1483.997660
2270  2022-04-06      22021  1612.407026
2277  2022-04-07      22021  1622.406408
2309  2022-04-10      22021  1614.765002
3505  2023-04-05      22022  1630.556259
3521  2023-04-07      22022  1635.448055
3539  2023-04-09      22022  1640.134739
4736  2024-04-11      22023  1722.465773
4751  2024-04-12      22023  1706.280424
4768  2024-04-14      22023  1707.417335
5963  2025-04-09      22024  1712.584917
5983  2025-04-11      22024  1698.866888
5996  2025-04-13      22024  1699.758097
7196  2026-04-09      22025  1680.685807
7210  2026-04-10      22025  1671.938094
7224  2026-04-12      22025  1673.902686
7230  2026-10-20      22026  1633.589338
7256  2026-10-23      22026          NaN
7276  2026-10-26      22026          NaN
---
       GAME_DATE  SEASON_ID     team_elo
12    2020-12-23      22020  1500.000000
16    2020-1

Evolved before production. Three things changed:

- **Margin of victory.** This version updates ratings by win/loss alone. The
  final `calculate_elo` in `feature_engineering.py` scales the update by a
  538-style margin-of-victory multiplier, so a blowout moves ratings more
  than a one-possession game.
- **Tuned constants.** `home_advantage=100`, `season_regression=0.75` here
  were guesses. `feature_engineering.py`'s `tune_elo` grid-searches `k`,
  `home_advantage` and `season_regression` against validation log loss and
  landed on `k=20`, `home_advantage=50`, `season_regression=0.6`.
- **Symmetric updates.** The update here computes the winner's and loser's
  new ratings from two separately-derived expressions; production applies
  one signed `change` to both teams, so a rating point gained by one side is
  always a point lost by the other.

## 4. Streak

Win/loss streak before each game, signed (positive = winning streak,
negative = losing streak), reset at the start of each season.

In [10]:
def add_streak(df):
    df = df.sort_values(['TEAM_ABBREVIATION', 'SEASON_ID', 'GAME_DATE']).reset_index(drop=True)

    streak_before_game = []
    current_streak = 0
    current_team = 0
    current_season = None

    for idx, row in df.iterrows():
        team = row['TEAM_ABBREVIATION']
        season = row['SEASON_ID']
        if team != current_team or season != current_season:
            current_streak = 0

        streak_before_game.append(current_streak)

        if row['WL'] == 'W':
            if current_streak >= 0:
                current_streak += 1
            else:
                current_streak = 1
        elif row['WL'] == 'L':
            if current_streak <= 0:
                current_streak -= 1
            else:
                current_streak = -1

        current_team = team
        current_season = season

        pass

    df['streak'] = streak_before_game
    return df
    

In [11]:
df_sorted = add_streak(df_sorted)

print(df_sorted[['TEAM_ABBREVIATION', 'WL', 'streak']].head(15))

   TEAM_ABBREVIATION WL  streak
0                ATL  W       0
1                ATL  W       1
2                ATL  W       2
3                ATL  L       3
4                ATL  W      -1
5                ATL  L       1
6                ATL  L      -1
7                ATL  L      -2
8                ATL  L      -3
9                ATL  W      -4
10               ATL  L       1
11               ATL  L      -1
12               ATL  W      -2
13               ATL  W       1
14               ATL  W       2


**Copied into production verbatim.** `add_streak` in `feature_engineering.py`
is this function unchanged - including the stray `pass` at the end of the
loop body, left over from an earlier draft.